# Lab 13 — A fair learner comparison — which classifier really wins on a new subject?

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. The instructor solution lives in `labs/solutions/` and is not included here.

**Covers.** Chapter 13 — §13.3–§13.4 (four ways to draw a fence; three learners on the same folds).

**Biomedical question.** Which classifier is really best for new subjects — and did I compare them fairly?
**Task type (§1.8).** Classification — honest model comparison to a new-subject claim.
**Information that must be preserved.** *two* things at once — the **claim** (each score must represent an unseen subject) **and** the **fairness** (every model scored on the SAME folds).
**Main assumptions.** epochs from one subject share that subject's idiosyncratic offset, so subjects stay whole across the split (GroupKFold).
**Primary diagnostic.** subject-independent Cohen's kappa on shared folds, read against the majority-class baseline; plus the optimism gap (resubstitution − subject-independent).
**Transfer challenge.** does the winner — and its margin over the baseline — survive on a new device or clinical site?

*Self-contained: a seeded synthetic multi-subject feature cohort, no data files, no `bsp`. Runs fully offline in well under a minute. Theme (§1.8): there is usually **no single best** method — several learners tie across these folds — **but wrong choices still exist** (an unfair comparison, or reporting a resubstitution score as if it were skill).*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab13_fair_learner_comparison/lab13_fair_learner_comparison.ipynb) [![nbviewer](https://img.shields.io/badge/view-nbviewer-orange)](https://nbviewer.org/github/farhad-abtahi/CM2013/blob/main/labs/lab13_fair_learner_comparison/lab13_fair_learner_comparison.ipynb) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab13_fair_learner_comparison.ipynb)

In [ ]:
# --- shared setup (reproducible; offline synthetic cohort) ---
import numpy as np, matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, GroupKFold
from sklearn.metrics import cohen_kappa_score, accuracy_score
rng = np.random.default_rng(2013)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

CLASSES = ["rest", "left", "right", "feet"]   # e.g. a 4-class motor-imagery task

def make_cohort(n_subj=10, per_class=45, sep=1.6, within=1.1, offset=1.5, n_feat=6, seed=2013):
    """A synthetic multi-subject feature cohort with THREE deliberate ingredients:
      * class OVERLAP     - within-class noise (`within`) is large enough that NO learner is perfect;
      * a per-SUBJECT OFFSET - every epoch of subject s is shifted by the SAME random vector
                            (`offset`): a physiological / electrode nuisance a model can memorise on
                            the training subjects but that is DIFFERENT for an unseen subject;
      * whole SUBJECTS    - returned in `groups`, so a subject-independent split can hold them out.
    The offset is why a random-row split would lie and a subject-grouped split tells the truth: a
    learner that keys on the offset scores well in-sample yet cannot transfer to a new subject."""
    r = np.random.default_rng(seed)
    K = len(CLASSES)
    centres = r.normal(0.0, sep, size=(K, n_feat))           # fixed class prototypes (deterministic)
    X, y, g = [], [], []
    for s in range(n_subj):
        subj_offset = r.normal(0.0, offset, size=n_feat)      # per-subject nuisance shift (the trap)
        for c in range(K):
            for _ in range(per_class):
                X.append(centres[c] + subj_offset + r.normal(0.0, within, size=n_feat))
                y.append(c); g.append(s)
    return np.array(X), np.array(y), np.array(g)

In [ ]:
X, y, groups = make_cohort()
print(f"cohort: {X.shape[0]} epochs x {X.shape[1]} features, "
      f"{len(np.unique(groups))} subjects, classes = {CLASSES}")
print("per-class counts:", np.bincount(y))

## 1. The trap — a score that only looks perfect

Fit **one** `RandomForestClassifier` on **all** the data, then score it on the **same** rows it trained on (*resubstitution*). It hits ~1.00 — and that number is worthless as evidence: the model is allowed to look up the answer it memorised, so this measures **memory**, not **skill on a new subject**. Reporting it as "the result" is the classic wrong move.

In [ ]:
rf_all = RandomForestClassifier(n_estimators=300, random_state=0).fit(X, y)
resub_pred  = rf_all.predict(X)                    # predict the TRAINING rows -> it has already seen them
resub_acc   = accuracy_score(y, resub_pred)
resub_kappa = cohen_kappa_score(y, resub_pred)
print(f"resubstitution accuracy = {resub_acc:.3f}   # looks perfect...")
print(f"resubstitution kappa    = {resub_kappa:.3f}")
print("^ NOT skill: the forest memorised these rows. It says nothing about an UNSEEN subject.")

## 2. A fair comparison — the SAME folds, subject-independent kappa

The real question: of **k-NN**, an **SVM**, and a **RandomForest**, which is best on a subject it has never seen? Fairness has one rule — **every model is scored on the identical folds**. Build the folds **once** with `GroupKFold` (whole subjects held out) and reuse them for all three via `cross_val_predict`. Score with **Cohen's kappa** (chance-corrected, so it is honest under class imbalance and directly comparable to the baseline in §3).

In [ ]:
# TODO evaluate k-NN (a StandardScaler + KNeighbors pipeline), SVM (a StandardScaler + SVC
# pipeline) and RandomForest on the SAME GroupKFold folds via cross_val_predict, and score
# each with Cohen's kappa. Build ONE cv object and ONE `models` dict, then loop -> that is
# what keeps the comparison fair (identical folds AND identical preprocessing for all).
# NB: GroupKFold(5) is subject-INDEPENDENT 5-fold (whole subjects held out), NOT leave-one-subject-out
# (that would be LeaveOneGroupOut, one fold per subject); with 10 subjects we use 5 folds.
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: why must all three share ONE cv object? What could a lucky learner do with its own
# private split? And why Cohen's kappa here rather than raw accuracy?

## 3. The baseline you must beat

A kappa only means something against a floor. The cheapest honest floor is the **majority-class** predictor (always guess the commonest label): by construction its **kappa ≈ 0**. Every real model must clear it — a learner that cannot is worse than a constant. Then read the winner *with a paired eye*: compare the models **fold-by-fold on the same folds** and ask whether the gap between them is bigger than the fold-to-fold wobble, or just noise.

In [ ]:
# TODO (a) compute the majority-class kappa (~0) and confirm each model beats it;
#      (b) name the nominal winner and its margin over the runner-up;
#      (c) take a PAIRED view on the shared folds: per-fold kappa for each model, then compare the
#          winner's per-fold lead to the fold-to-fold spread -> is the lead consistent, or does it flip across folds?
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: all three clear the baseline, but is the winner's lead larger than the fold-to-fold
# std? State the honest headline: is there a single best model here, or a tie above a clear floor?

## 4. Live sanity check — the optimism gap

Tie the lab together with one assertion: for the **same forest**, the resubstitution score (§1) must sit far above the subject-independent score (§2). That gap **is** the optimism the trap hid; if it were small, the split would not really be testing transfer to a new subject.

In [ ]:
grouped_forest = kappas["RandomForest"]
optimism_gap   = resub_kappa - grouped_forest
print(f"forest resubstitution kappa      = {resub_kappa:.3f}   (§1, memorised rows)")
print(f"forest subject-independent kappa = {grouped_forest:.3f}   (§2, unseen subjects)")
print(f"optimism gap                     = {optimism_gap:.3f}")
assert resub_kappa > grouped_forest + 0.2, "expected resubstitution >> subject-independent"
print("sanity check PASSED: resubstitution >> subject-independent  (the gap the trap hid)")

## Reflection

This reflection is for your own practice — there is nothing to submit. What matters is the *reasoning*, not which model came out nominally first.

1. **Stable vs changed.** Going from §1 (resubstitution) to the fair subject-independent comparison in §2–§3, which conclusion **stayed stable** and which **changed**? (Hint: separate the *level* of the scores from the *ranking* of the models.)
2. **Winner or tie?** State your headline: is there a single best classifier here, or a tie above the baseline? Quote the paired per-fold numbers (winner's margin vs the fold-to-fold std) as evidence.
3. **New device / site.** What evidence would you need before claiming this winner is best on a NEW recording device or clinical site — which axis of the split changes, and what would you re-measure?

**Rule out (name the wrong comparison).** Give one concrete *wrong* way to run this study — e.g. scoring k-NN, SVM and RF on **different** splits (each on its own lucky fold), or reporting the §1 **resubstitution** accuracy as "the result" — and name the requirement it breaks: it violates the **§1.8 worldview** that a score must (a) represent the *claim* (an unseen subject) and (b) be *fair* (the same folds for every model). Tie it to the theme: there is **no single best** learner here — several tie across these grouped folds — **but wrong choices still exist**, and an unfair or in-sample comparison is one of them.

> *Your answers here.*

---
*Type-2 lab for **Biomedical Signal Processing & Data Analytics**. Synthetic cohort; illustrative numbers. The lesson is the method, not the winner: compare fairly (same folds), report against a baseline, and claim only what the split can support.*